**Chatbot prototype with full NLP pipeline**

In [1]:
# loading the liberaries
import re
import os
import json
import pickle
import logging
import pandas as pd
import numpy as np
from datetime import datetime
 
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
from sklearn.pipeline import Pipeline

In [5]:
# paths
BASE = os.getcwd()
DATA_PATH   = os.path.join(BASE, "data", "chatbot_dataset.csv")
MODEL_PATH  = os.path.join(BASE, "models", "chatbot_model.pkl")
LOG_PATH    = os.path.join(BASE, "logs", "interactions.json")
METRICS_PATH = os.path.join(BASE, "logs", "model_metrics.json")
 
os.makedirs(os.path.join(BASE, "logs"), exist_ok=True)

In [6]:
# NLKT setup
for pkg in ["punkt", "stopwords", "wordnet", "averaged_perceptron_tagger", "punkt_tab"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)
 
STOP_WORDS = set(stopwords.words("english")) - {
    "not", "no", "can", "cannot", "don't", "isn't", "aren't", "won't"
}
lemmatizer = WordNetLemmatizer()

In [7]:
# NLP Preprocessing 
def preprocess(text: str) -> str:
    """
    Full NLP pipeline:
      1. Lowercase
      2. Remove special chars / numbers
      3. Tokenize
      4. Remove stopwords
      5. Lemmatize
    """
    text = text.lower().strip()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return " ".join(tokens)

In [8]:
#  Intent → Rich Response Map
RESPONSES = {
    "Greeting": [
        "👋 Hello! Welcome to EduBot — your AI assistant for our online learning platform. How can I help you today?",
        "Hi there! 😊 I'm EduBot, ready to assist you with courses, enrollment, payments, and more!",
        "Hey! Great to see you here. Ask me anything about our programs or platform support.",
    ],
    "Farewell": [
        "Goodbye! 🎓 Best of luck with your learning journey. Come back anytime!",
        "See you soon! Don't hesitate to reach out if you have more questions. Happy learning! 📚",
        "Take care! We hope to see you again. Keep learning and growing! 🚀",
    ],
    "Course_Info": [
        (
            "📚 We offer a wide range of courses:\n\n"
            "• **Data Science** (6 months) — Python, ML, Statistics\n"
            "• **Artificial Intelligence** (6 months) — Neural Networks, NLP, CV\n"
            "• **Machine Learning** (4 months) — Supervised/Unsupervised Learning\n"
            "• **Business Analytics** (3 months) — Power BI, SQL, Excel\n"
            "• **Digital Marketing** (2 months) — SEO, SEM, Social Media\n"
            "• **Cloud Computing** (4 months) — AWS, Azure, GCP\n"
            "• **Cyber Security** (5 months) — Ethical Hacking, VAPT\n"
            "• **Web Development** (3 months) — React, Node.js, MongoDB\n\n"
            "All courses include live classes, recorded sessions, projects & certificate. "
            "Would you like details on a specific course?"
        ),
    ],
    "Enrollment": [
        (
            "📝 **Enrollment Process:**\n\n"
            "1️⃣ Visit our website and click **'Enroll Now'**\n"
            "2️⃣ Create your account or log in\n"
            "3️⃣ Select your desired course\n"
            "4️⃣ Complete payment (UPI / Card / EMI available)\n"
            "5️⃣ Receive confirmation email within 24 hours\n\n"
            "**Required Documents:** Government ID, Educational Certificate, Passport Photo\n"
            "Enrollment is open year-round. Need help with a specific step?"
        ),
    ],
    "Payment": [
        (
            "💳 **Payment Options:**\n\n"
            "✅ UPI (GPay, PhonePe, Paytm)\n"
            "✅ Credit & Debit Cards (Visa, Mastercard, RuPay)\n"
            "✅ Net Banking (all major banks)\n"
            "✅ EMI (0% interest, 3/6/12 months via HDFC, ICICI, SBI cards)\n"
            "✅ International payments via PayPal\n\n"
            "**Refund Policy:** 100% refund within 7 days of enrollment.\n"
            "For payment issues, email: payments@eduplatform.com or call 1800-XXX-XXXX."
        ),
    ],
    "Technical_Support": [
        (
            "🔧 **Technical Support:**\n\n"
            "Common fixes:\n"
            "• **Login issues** → Clear browser cache → Try incognito mode → Reset password\n"
            "• **Video not playing** → Check internet speed → Update browser → Disable ad-blocker\n"
            "• **App crashes** → Update to latest version → Reinstall app\n"
            "• **Audio issues** → Check device volume → Use Chrome/Firefox for best experience\n\n"
            "📧 support@eduplatform.com\n"
            "📞 1800-XXX-XXXX (Mon–Sat, 9AM–6PM)\n"
            "💬 Live chat available on our website"
        ),
    ],
    "Certificate": [
        (
            "🏆 **Certificate Information:**\n\n"
            "• Certificate issued upon **course completion** (minimum 80% score)\n"
            "• **Industry-recognized** and shareable on LinkedIn\n"
            "• Available as **digital PDF** (instant) and **physical copy** (7–10 days)\n"
            "• Verified via unique QR code on certificate\n"
            "• Joint certification with **Amity University / NSDC** (select courses)\n\n"
            "Download from: My Profile → Certificates → Download\n"
            "Issues? Email: certificates@eduplatform.com"
        ),
    ],
    "Placement_Career": [
        (
            "💼 **Placement & Career Support:**\n\n"
            "✅ Dedicated placement cell with **500+ hiring partners**\n"
            "✅ Resume building & LinkedIn profile optimization\n"
            "✅ Mock interviews with industry experts\n"
            "✅ Access to exclusive job portal for alumni\n"
            "✅ Internship opportunities during the course\n"
            "✅ Career counseling sessions (1-on-1)\n\n"
            "**Average Package:** ₹6–12 LPA\n"
            "**Top Recruiters:** TCS, Infosys, Wipro, Amazon, Deloitte, KPMG\n"
            "**Placement Rate:** 85% within 3 months of completion"
        ),
    ],
    "fallback": [
        (
            "🤔 I'm sorry, I didn't quite understand that. Could you rephrase your question?\n\n"
            "I can help you with:\n"
            "• 📚 Course information\n"
            "• 📝 Enrollment & registration\n"
            "• 💳 Payments & fees\n"
            "• 🔧 Technical support\n"
            "• 🏆 Certificates\n"
            "• 💼 Placements & career\n\n"
            "Please ask about any of these topics!"
        ),
    ],
} 

In [9]:
# Model Training
def train_model():
    df = pd.read_csv(DATA_PATH)
    df["Processed"] = df["Query"].apply(preprocess)
 
    X, y = df["Processed"], df["Intent"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
 
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=5000,
            sublinear_tf=True,
            min_df=1,
        )),
        ("clf", LogisticRegression(
            C=5.0,
            max_iter=1000,
            solver="lbfgs",
            random_state=42,
        )),
    ])
 
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
 
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="f1_macro")
 
    metrics = {
        "accuracy":       round(accuracy_score(y_test, y_pred), 4),
        "f1_macro":       round(f1_score(y_test, y_pred, average="macro"), 4),
        "cv_mean":        round(cv_scores.mean(), 4),
        "cv_std":         round(cv_scores.std(), 4),
        "cv_scores":      [round(s, 4) for s in cv_scores.tolist()],
        "classes":        list(pipeline.classes_),
        "train_samples":  len(X_train),
        "test_samples":   len(X_test),
        "report":         classification_report(y_test, y_pred, output_dict=True),
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=list(pipeline.classes_)).tolist(),
        "trained_at":     datetime.now().isoformat(),
    }
 
    with open(MODEL_PATH, "wb") as f:
        pickle.dump(pipeline, f)
    with open(METRICS_PATH, "w") as f:
        json.dump(metrics, f, indent=2)
 
    print(f"✅ Model trained | Accuracy: {metrics['accuracy']} | F1: {metrics['f1_macro']}")
    print(f"   CV Mean: {metrics['cv_mean']} ± {metrics['cv_std']}")
    return pipeline, metrics
 
 
def load_model():
    if not os.path.exists(MODEL_PATH):
        return train_model()[0]
    with open(MODEL_PATH, "rb") as f:
        return pickle.load(f)

In [10]:
# Chatbot Core
class EduBot:
    CONFIDENCE_THRESHOLD = 0.45
 
    def __init__(self):
        self.model = load_model()
        self.interactions = self._load_logs()
 
    def _load_logs(self):
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH) as f:
                return json.load(f)
        return []
 
    def _save_logs(self):
        with open(LOG_PATH, "w") as f:
            json.dump(self.interactions, f, indent=2)
 
    def get_response(self, user_input: str, session_id: str = "default") -> dict:
        processed = preprocess(user_input)
        proba = self.model.predict_proba([processed])[0]
        classes = self.model.classes_
        confidence = float(np.max(proba))
        intent = classes[np.argmax(proba)]
 
        if confidence < self.CONFIDENCE_THRESHOLD:
            intent = "fallback"
            response = random.choice(RESPONSES["fallback"])
        else:
            response = random.choice(RESPONSES.get(intent, RESPONSES["fallback"]))
 
        # All-class confidence breakdown
        conf_breakdown = {c: round(float(p), 4) for c, p in zip(classes, proba)}
 
        log_entry = {
            "id":               len(self.interactions) + 1,
            "session_id":       session_id,
            "timestamp":        datetime.now().isoformat(),
            "user_input":       user_input,
            "processed_input":  processed,
            "predicted_intent": intent,
            "confidence":       round(confidence, 4),
            "conf_breakdown":   conf_breakdown,
            "response":         response,
            "rating":           None,
            "feedback":         None,
        }
        self.interactions.append(log_entry)
        self._save_logs()
 
        return {
            "intent":     intent,
            "confidence": confidence,
            "response":   response,
            "log_id":     log_entry["id"],
            "breakdown":  conf_breakdown,
        }
 
    def submit_rating(self, log_id: int, rating: int, feedback: str = ""):
        for entry in self.interactions:
            if entry["id"] == log_id:
                entry["rating"] = rating
                entry["feedback"] = feedback
                break
        self._save_logs()
 
 
import random

**Chatbot testing**

In [15]:
# Chatbot testing
bot = EduBot()

print("EduBot Started (type 'exit' to stop)\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Goodbye!")
        break

    result = bot.get_response(user_input)

    print("Bot:", result["response"])
    print("Confidence:", round(result["confidence"], 2))

EduBot Started (type 'exit' to stop)



You:  Hey there?


Bot: Hi there! 😊 I'm EduBot, ready to assist you with courses, enrollment, payments, and more!
Confidence: 0.87


You:  which corses you are offering?


Bot: 🤔 I'm sorry, I didn't quite understand that. Could you rephrase your question?

I can help you with:
• 📚 Course information
• 📝 Enrollment & registration
• 💳 Payments & fees
• 🔧 Technical support
• 🏆 Certificates
• 💼 Placements & career

Please ask about any of these topics!
Confidence: 0.36


...

Bot: 💳 **Payment Options:**

✅ UPI (GPay, PhonePe, Paytm)
✅ Credit & Debit Cards (Visa, Mastercard, RuPay)
✅ Net Banking (all major banks)
✅ EMI (0% interest, 3/6/12 months via HDFC, ICICI, SBI cards)
✅ International payments via PayPal

**Refund Policy:** 100% refund within 7 days of enrollment.
For payment issues, email: payments@eduplatform.com or call 1800-XXX-XXXX.
Confidence: 0.65


You:  from where I will get my certificate?


Bot: 🏆 **Certificate Information:**

• Certificate issued upon **course completion** (minimum 80% score)
• **Industry-recognized** and shareable on LinkedIn
• Available as **digital PDF** (instant) and **physical copy** (7–10 days)
• Verified via unique QR code on certificate
• Joint certification with **Amity University / NSDC** (select courses)

Download from: My Profile → Certificates → Download
Issues? Email: certificates@eduplatform.com
Confidence: 0.87


You:  exit


Bot: Goodbye!
